# AgriLink — Local-Popularity Recommendation

# AgriLink Customer Recommendation
This notebook is a standalone baseline. It runs with generated example data by default.
To use AgriLink data, set `USE_CSV = True` and update `CSV_PATH`. Expected columns are documented below.
Do not report results from synthetic data as real AgriLink performance.

**Goal:** recommend products popular among customers in the same locality.
**Required real-data columns:** `CustomerID`, `OrderID`, `Product`, `Quantity`, `Location`. If location is stored in a separate customer table, merge it into order lines before running.

In [ ]:
import pandas as pd
import numpy as np

USE_CSV = False
CSV_PATH = "/content/agrilink_orders_with_location.csv"
rng = np.random.default_rng(14)
products = ["Apple","Banana","Beans","Brinjal","Cabbage","Carrot","Coriander",
            "Cucumber","Green Chilli","Onion","Orange","Papaya","Potato","Rice","Tomato"]
locations = ["North","South","East","West"]
rows=[]
for c in range(1,101):
    loc = locations[(c-1) % len(locations)]
    # Give each locality a few stronger preferences to create a demonstrable example.
    local_pref = {
        "North":["Apple","Carrot","Potato","Onion"],
        "South":["Banana","Rice","Tomato","Coriander"],
        "East":["Cabbage","Brinjal","Green Chilli","Potato"],
        "West":["Orange","Papaya","Cucumber","Beans"]
    }[loc]
    pool = list(dict.fromkeys(local_pref + products))
    for o in range(1,6):
        for p in rng.choice(pool, size=3, replace=False):
            rows.append({"CustomerID":c,"OrderID":f"{c}_{o}","Product":p,
                         "Quantity":int(rng.integers(1,4)),"Location":loc})
df = pd.DataFrame(rows)
if USE_CSV:
    df = pd.read_csv(CSV_PATH)
    required = {"CustomerID","OrderID","Product","Quantity","Location"}
    missing = required - set(df.columns)
    if missing: raise ValueError(f"CSV is missing columns: {missing}")
df["Location"] = df["Location"].astype(str).str.strip()
df.head()

## Rank products within a locality
Use distinct orders as the popularity count so that one large-quantity order does not dominate the ranking. Optionally exclude products already purchased by the target customer.

In [ ]:
def local_popularity_recommendations(customer_id, df, top_n=10,
                                     exclude_purchased=False):
    customer_rows = df[df["CustomerID"] == customer_id]
    if customer_rows.empty:
        return pd.DataFrame(columns=["Product","LocalOrderCount","LocalCustomerCount"])
    location = customer_rows["Location"].mode().iloc[0]
    local_df = df[df["Location"] == location]

    scored = local_df.groupby("Product").agg(
        LocalOrderCount=("OrderID","nunique"),
        LocalCustomerCount=("CustomerID","nunique"),
        TotalQuantity=("Quantity","sum")
    ).reset_index()

    if exclude_purchased:
        bought = set(customer_rows["Product"])
        scored = scored[~scored["Product"].isin(bought)]

    scored = scored.sort_values(
        ["LocalOrderCount","LocalCustomerCount","TotalQuantity"],
        ascending=False
    ).head(top_n).reset_index(drop=True)
    scored.insert(0, "Location", location)
    return scored

customer_id = df["CustomerID"].iloc[0]
display(local_popularity_recommendations(customer_id, df, top_n=10))

## Compare local and global popularity

In [ ]:
def global_popularity(df, top_n=10):
    return (df.groupby("Product")
              .agg(GlobalOrderCount=("OrderID","nunique"),
                   GlobalCustomerCount=("CustomerID","nunique"))
              .sort_values(["GlobalOrderCount","GlobalCustomerCount"], ascending=False)
              .head(top_n).reset_index())

print("Local recommendations")
display(local_popularity_recommendations(customer_id, df, top_n=10))
print("Global popularity baseline")
display(global_popularity(df, top_n=10))

## Limitations
- Location must be reliable and appropriately coarse-grained (for example, service zone or locality); avoid exposing precise customer addresses.
- If a locality has little data, fall back to broader-area or global popularity.
- For rigorous evaluation, calculate popularity from training-period orders only, then test on later orders to prevent future-data leakage.